# EL-GHALI MOHAMED


# Implémentation d'un XGBoost simple du Régression



**Les formules clés de XGBoost :**
 XGBoost utilise le **Score de Similarité** pour regrouper les erreurs et calculer le **Gain** d'une division.

* **Gradient ($g_i$)** : L'erreur de prédiction pour une ligne. (Pour l'erreur quadratique : $g_i = \hat{y}_i - y_i$)
* **Hessien ($h_i$)** : La dérivée de l'erreur. (Pour l'erreur quadratique : $h_i = 1$)
* **Score de Similarité** :
$$Similarity = \frac{(\sum g_i)^2}{\sum h_i + \lambda}$$


* **Gain** :
$$Gain = Similarity_{gauche} + Similarity_{droite} - Similarity_{racine}$$


* **Valeur de sortie d'une feuille** :
$$Output = - \frac{\sum g_i}{\sum h_i + \lambda}$$





In [3]:
import pandas as pd
import numpy as np

# DATASET
# On veut prédire le Salaire (en k€) en fonction de l'Âge
data = {
    'Age': [22, 25, 28, 35, 42, 45, 55, 60],
    'Salaire': [30, 35, 45, 65, 75, 80, 100, 110]
}
df = pd.DataFrame(data)
print("DATASET")
print(df)

DATASET
   Age  Salaire
0   22       30
1   25       35
2   28       45
3   35       65
4   42       75
5   45       80
6   55      100
7   60      110


### Les Calcules Mathématique de XGBoost
Nous définissons les fonctions qui calculent les Gradients, les Hessiens, la Similarité et le Gain. 

In [4]:
def calcul_similarite(gradients, hessiens, reg_lambda):
    """Calcule le score de similarité d'un nœud selon la formule XGBoost."""
    somme_g = sum(gradients)
    somme_h = sum(hessiens)
    # Formule : (Somme des Gradients)^2 / (Somme des Hessiens + Lambda)
    return (somme_g ** 2) / (somme_h + reg_lambda)

def calcul_gain(g_gauche, h_gauche, g_droite, h_droite, g_racine, h_racine, reg_lambda):
    """Calcule le gain d'une séparation (split)."""
    sim_gauche = calcul_similarite(g_gauche, h_gauche, reg_lambda)
    sim_droite = calcul_similarite(g_droite, h_droite, reg_lambda)
    sim_racine = calcul_similarite(g_racine, h_racine, reg_lambda)

    # On cherche à maximiser cette valeur !
    return sim_gauche + sim_droite - sim_racine

def valeur_feuille(gradients, hessiens, reg_lambda):
    """Calcule la valeur de prédiction finale pour une feuille de l'arbre."""
    # Formule : - (Somme des Gradients) / (Somme des Hessiens + Lambda)
    return - sum(gradients) / (sum(hessiens) + reg_lambda)

### Construction d'un Arbre XGBoost


In [5]:
def trouver_meilleur_split(df_actuel, feature, reg_lambda):
    """Trouve le meilleur seuil numérique pour diviser une caractéristique."""
    valeurs_triees = sorted(df_actuel[feature].unique())
    meilleur_gain = 0
    meilleur_seuil = None

    g_racine = df_actuel['Gradient'].tolist()
    h_racine = df_actuel['Hessien'].tolist()

    # On teste les seuils entre chaque valeur adjacente
    for i in range(len(valeurs_triees) - 1):
        seuil = (valeurs_triees[i] + valeurs_triees[i+1]) / 2.0

        # Séparation des données
        df_gauche = df_actuel[df_actuel[feature] < seuil]
        df_droite = df_actuel[df_actuel[feature] >= seuil]

        # Récupération des gradients et hessiens
        g_g = df_gauche['Gradient'].tolist()
        h_g = df_gauche['Hessien'].tolist()
        g_d = df_droite['Gradient'].tolist()
        h_d = df_droite['Hessien'].tolist()

        # Calcul du gain pour ce seuil
        gain = calcul_gain(g_g, h_g, g_d, h_d, g_racine, h_racine, reg_lambda)

        if gain > meilleur_gain:
            meilleur_gain = gain
            meilleur_seuil = seuil

    return meilleur_gain, meilleur_seuil

def construire_arbre_xgboost(df_actuel, features, reg_lambda):
    """Construit un petit arbre (profondeur 1) basé sur le meilleur gain."""
    meilleur_gain_global = 0
    meilleilleure_feature = None
    meilleur_seuil_global = None

    #  Chercher la meilleure séparation
    for feature in features:
        gain, seuil = trouver_meilleur_split(df_actuel, feature, reg_lambda)
        if gain > meilleur_gain_global:
            meilleur_gain_global = gain
            meilleilleure_feature = feature
            meilleur_seuil_global = seuil

    # S'il n'y a aucun gain positif, on ne divise pas
    if meilleur_gain_global == 0 or meilleur_seuil_global is None:
        return None

    #  Calculer les valeurs des feuilles
    df_gauche = df_actuel[df_actuel[meilleilleure_feature] < meilleur_seuil_global]
    df_droite = df_actuel[df_actuel[meilleilleure_feature] >= meilleur_seuil_global]

    valeur_gauche = valeur_feuille(df_gauche['Gradient'].tolist(), df_gauche['Hessien'].tolist(), reg_lambda)
    valeur_droite = valeur_feuille(df_droite['Gradient'].tolist(), df_droite['Hessien'].tolist(), reg_lambda)

    # On retourne la structure de l'arbre
    return {
        'feature': meilleilleure_feature,
        'seuil': meilleur_seuil_global,
        'valeur_gauche': valeur_gauche,
        'valeur_droite': valeur_droite
    }

def predire_arbre_xgboost(requete, arbre):
    """Fait passer une ligne de donnée dans notre arbre pour obtenir la prédiction."""
    if arbre is None:
        return 0
    valeur_feature = requete[arbre['feature']]
    if valeur_feature < arbre['seuil']:
        return arbre['valeur_gauche']
    else:
        return arbre['valeur_droite']

### La Boucle  d'Apprentissage (Boosting)

In [6]:
def entrainer_xgboost(df_train, features, cible, nb_arbres=3, learning_rate=0.3, reg_lambda=1.0):
    print("Début de l'entraînement XGBoost...")
    foret = []

    #  Prédiction initiale (La moyenne de la cible)
    prediction_initiale = df_train[cible].mean()
    df_train['Prediction'] = prediction_initiale

    for i in range(nb_arbres):
        print(f"\n--- Entraînement de l'Arbre {i+1} ---")

        #  Calcul des Gradients et Hessiens
        df_train['Gradient'] = df_train['Prediction'] - df_train[cible]
        # Hessien = 1 (Pour la régression de l'erreur quadratique)
        df_train['Hessien'] = 1

        #  Construction de l'arbre sur les erreurs actuelles
        arbre = construire_arbre_xgboost(df_train, features, reg_lambda)
        foret.append(arbre)

        print(f"Arbre créé : Si {arbre['feature']} < {arbre['seuil']}")
        print(f"  -> Branche Gauche : ajouter {arbre['valeur_gauche']:.2f}")
        print(f"  -> Branche Droite : ajouter {arbre['valeur_droite']:.2f}")

        #  Mise à jour des prédictions de notre dataset
        predictions_arbre = df_train.apply(lambda row: predire_arbre_xgboost(row, arbre), axis=1)

        # Nouvelle Prédiction = Ancienne Prédiction + (Learning Rate * Sortie de l'arbre)
        df_train['Prediction'] = df_train['Prediction'] + (learning_rate * predictions_arbre)

        # Affichage de l'erreur moyenne pour voir si le modèle s'améliore
        erreur_moyenne = abs(df_train['Prediction'] - df_train[cible]).mean()
        print(f"Erreur moyenne absolue après cet arbre : {erreur_moyenne:.2f} k€")

    return foret, prediction_initiale

#  LANCEMENT DE L'ALGORITHME
features = ['Age']
cible = 'Salaire'

# On crée 5 arbres successifs
model_xgboost, prediction_base = entrainer_xgboost(
    df.copy(), features, cible,
    nb_arbres=5,
    learning_rate=0.3,
    reg_lambda=1.0
)

print("\n=== TEST DE PRÉDICTION ===")
#  prédire pour une nouvelle personne de 30 ans
nouveau_profil = {'Age': 30}

# On part de la prédiction de base
prediction_finale = prediction_base

# On ajoute le résultat de chaque arbre de notre forêt
for i, arbre in enumerate(model_xgboost):
    ajustement = predire_arbre_xgboost(nouveau_profil, arbre)
    prediction_finale += (0.3 * ajustement) # 0.3 est notre learning_rate
    print(f"L'arbre {i+1} ajuste le salaire de : {0.3 * ajustement:+.2f} k€")

print(f"\nPrédiction finale pour 30 ans : {prediction_finale:.2f} k€")

Début de l'entraînement XGBoost...

--- Entraînement de l'Arbre 1 ---
Arbre créé : Si Age < 38.5
  -> Branche Gauche : ajouter -19.00
  -> Branche Droite : ajouter 19.00
Erreur moyenne absolue après cet arbre : 18.85 k€

--- Entraînement de l'Arbre 2 ---
Arbre créé : Si Age < 31.5
  -> Branche Gauche : ajouter -18.85
  -> Branche Droite : ajouter 12.57
Erreur moyenne absolue après cet arbre : 15.01 k€

--- Entraînement de l'Arbre 3 ---
Arbre créé : Si Age < 50.0
  -> Branche Gauche : ajouter -8.28
  -> Branche Droite : ajouter 18.69
Erreur moyenne absolue après cet arbre : 12.97 k€

--- Entraînement de l'Arbre 4 ---
Arbre créé : Si Age < 31.5
  -> Branche Gauche : ajouter -12.75
  -> Branche Droite : ajouter 8.80
Erreur moyenne absolue après cet arbre : 10.60 k€

--- Entraînement de l'Arbre 5 ---
Arbre créé : Si Age < 43.5
  -> Branche Gauche : ajouter -7.06
  -> Branche Droite : ajouter 10.61
Erreur moyenne absolue après cet arbre : 8.51 k€

=== TEST DE PRÉDICTION ===
L'arbre 1 ajuste